# 5. Análise dos resultados e exportação para o Power BI
Este notebook resume a classificação pelos eixos da BNCC e organiza os dados em tabelas que podem ser importadas no Power BI.

In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.table import Table, TableStyleInfo

inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')

pasta_processados = raiz / 'dados' / '1_processados'
pasta_consumo = raiz / 'dados' / '2_consumo'
pasta_consumo.mkdir(parents=True, exist_ok=True)

## 5.1 Leitura dos resultados da classificação

In [ ]:
artigos = pd.read_csv(pasta_processados / '04_artigos_classificados_bncc.csv', encoding='utf-8-sig')
classificacoes = pd.read_csv(pasta_processados / '04_classificacoes_bncc.csv', encoding='utf-8-sig')
frequencia_descritores = pd.read_csv(pasta_processados / '04_frequencia_descritores_bncc.csv', encoding='utf-8-sig')
termos_titulos = pd.read_csv(pasta_processados / '03_termos_titulos.csv', encoding='utf-8-sig')
bigramas_titulos = pd.read_csv(pasta_processados / '03_bigramas_titulos.csv', encoding='utf-8-sig')
ranking_termos = pd.read_csv(pasta_processados / '03_ranking_termos_titulos.csv', encoding='utf-8-sig')
ranking_bigramas = pd.read_csv(pasta_processados / '03_ranking_bigramas_titulos.csv', encoding='utf-8-sig')
freq_termos_ano_evento = pd.read_csv(pasta_processados / '03_frequencia_termos_ano_evento.csv', encoding='utf-8-sig')
freq_bigramas_ano_evento = pd.read_csv(pasta_processados / '03_frequencia_bigramas_ano_evento.csv', encoding='utf-8-sig')

artigos['ano'] = pd.to_numeric(artigos['ano'], errors='coerce').astype('Int64')
classificacoes['ano'] = pd.to_numeric(classificacoes['ano'], errors='coerce').astype('Int64')
print(f'Artigos analisados: {len(artigos)}')
print(f'Relações artigo–eixo: {len(classificacoes)}')

## 5.2 Indicadores gerais

In [ ]:
total_artigos = len(artigos)
total_classificados = int((artigos['quantidade_eixos'] > 0).sum())
total_nao_classificados = total_artigos - total_classificados
cobertura = round(total_classificados / total_artigos * 100, 2)

indicadores = pd.DataFrame({
    'indicador': ['Artigos analisados', 'Artigos classificados', 'Artigos não classificados', 'Cobertura da classificação (%)'],
    'valor': [total_artigos, total_classificados, total_nao_classificados, cobertura],
})
indicadores

## 5.3 Distribuição por eixo

In [ ]:
resumo_eixos = (classificacoes.groupby('eixo_bncc')['id_artigo']
    .nunique().rename('quantidade_artigos').reset_index())
resumo_eixos['percentual_corpus'] = (resumo_eixos['quantidade_artigos'] / total_artigos * 100).round(2)
resumo_eixos = resumo_eixos.sort_values('quantidade_artigos', ascending=False)
resumo_eixos

In [ ]:
resumo_evento_eixo = (classificacoes.groupby(['evento', 'eixo_bncc'])['id_artigo']
    .nunique().rename('quantidade_artigos').reset_index())
resumo_evento_eixo.pivot(index='eixo_bncc', columns='evento', values='quantidade_artigos').fillna(0)

## 5.4 Evolução anual e cobertura

In [ ]:
resumo_ano_evento = (artigos.assign(classificado=artigos['quantidade_eixos'].gt(0))
    .groupby(['ano', 'evento'])
    .agg(total_artigos=('id_artigo', 'nunique'), artigos_classificados=('classificado', 'sum'))
    .reset_index())
resumo_ano_evento['artigos_nao_classificados'] = resumo_ano_evento['total_artigos'] - resumo_ano_evento['artigos_classificados']
resumo_ano_evento['cobertura_percentual'] = (resumo_ano_evento['artigos_classificados'] / resumo_ano_evento['total_artigos'] * 100).round(2)
resumo_ano_evento

In [ ]:
resumo_ano_evento_eixo = (classificacoes.groupby(['ano', 'evento', 'eixo_bncc'])['id_artigo']
    .nunique().rename('quantidade_artigos').reset_index())
resumo_ano_evento_eixo.head(12)

## 5.5 Sobreposição entre os eixos

In [ ]:
sobreposicao = (artigos['quantidade_eixos'].value_counts().sort_index()
    .rename_axis('quantidade_eixos').reset_index(name='quantidade_artigos'))
sobreposicao['percentual_corpus'] = (sobreposicao['quantidade_artigos'] / total_artigos * 100).round(2)
sobreposicao

## 5.6 Preparação do modelo de dados

In [ ]:
mapa_eixos = {
    'Pensamento Computacional': 'PC',
    'Mundo Digital': 'MD',
    'Cultura Digital': 'CD',
}

fato_artigos = artigos[['id_artigo', 'evento', 'ano', 'titulo', 'resumo', 'palavras_chave', 'url', 'quantidade_eixos']].copy()
fato_artigos['classificado_bncc'] = fato_artigos['quantidade_eixos'].gt(0)

ponte_artigo_eixo = classificacoes.rename(columns={
    'quantidade_evidencias': 'qtd_evidencias',
    'termos_encontrados': 'evidencias',
}).copy()
ponte_artigo_eixo.insert(1, 'id_eixo', ponte_artigo_eixo['eixo_bncc'].map(mapa_eixos))

dim_eixos = pd.DataFrame([
    {'id_eixo': 'PC', 'eixo_bncc': 'Pensamento Computacional', 'ordem': 1},
    {'id_eixo': 'MD', 'eixo_bncc': 'Mundo Digital', 'ordem': 2},
    {'id_eixo': 'CD', 'eixo_bncc': 'Cultura Digital', 'ordem': 3},
])
dim_eventos = pd.DataFrame({'evento': sorted(artigos['evento'].dropna().unique())})
dim_anos = pd.DataFrame({'ano': sorted(artigos['ano'].dropna().astype(int).unique())})

print(f'Artigos: {len(fato_artigos)}')
print(f'Relações artigo–eixo: {len(ponte_artigo_eixo)}')

## 5.7 Exportação dos arquivos XLSX

In [ ]:
def formatar_workbook(writer):
    for indice, planilha in enumerate(writer.book.worksheets, start=1):
        planilha.freeze_panes = 'A2'
        planilha.auto_filter.ref = planilha.dimensions
        for celula in planilha[1]:
            celula.fill = PatternFill('solid', fgColor='1F4E78')
            celula.font = Font(color='FFFFFF', bold=True)
            celula.alignment = Alignment(horizontal='center')
        for coluna in planilha.columns:
            valores = [str(c.value) if c.value is not None else '' for c in coluna[:200]]
            largura = min(max(max((len(v) for v in valores), default=0) + 2, 12), 60)
            planilha.column_dimensions[coluna[0].column_letter].width = largura
        if planilha.max_row > 1 and planilha.max_column > 0:
            tabela = Table(displayName=f'Tabela{indice}', ref=planilha.dimensions)
            tabela.tableStyleInfo = TableStyleInfo(name='TableStyleMedium2', showRowStripes=True)
            planilha.add_table(tabela)

arquivo_modelo = pasta_consumo / '05_modelo_power_bi_bncc.xlsx'
with pd.ExcelWriter(arquivo_modelo, engine='openpyxl') as writer:
    fato_artigos.to_excel(writer, sheet_name='Artigos', index=False)
    ponte_artigo_eixo.to_excel(writer, sheet_name='Artigo_Eixo', index=False)
    dim_eixos.to_excel(writer, sheet_name='Eixos', index=False)
    dim_eventos.to_excel(writer, sheet_name='Eventos', index=False)
    dim_anos.to_excel(writer, sheet_name='Anos', index=False)
    termos_titulos.to_excel(writer, sheet_name='Termos_Titulo', index=False)
    bigramas_titulos.to_excel(writer, sheet_name='Bigramas_Titulo', index=False)
    formatar_workbook(writer)

arquivo_resumos = pasta_consumo / '05_resumos_power_bi_bncc.xlsx'
with pd.ExcelWriter(arquivo_resumos, engine='openpyxl') as writer:
    indicadores.to_excel(writer, sheet_name='Indicadores', index=False)
    resumo_eixos.to_excel(writer, sheet_name='Resumo_Eixos', index=False)
    resumo_ano_evento.to_excel(writer, sheet_name='Ano_Evento', index=False)
    resumo_ano_evento_eixo.to_excel(writer, sheet_name='Ano_Evento_Eixo', index=False)
    resumo_evento_eixo.to_excel(writer, sheet_name='Evento_Eixo', index=False)
    sobreposicao.to_excel(writer, sheet_name='Sobreposicao', index=False)
    frequencia_descritores.to_excel(writer, sheet_name='Descritores_BNCC', index=False)
    ranking_termos.to_excel(writer, sheet_name='Ranking_Termos', index=False)
    ranking_bigramas.to_excel(writer, sheet_name='Ranking_Bigramas', index=False)
    freq_termos_ano_evento.to_excel(writer, sheet_name='Termos_Ano_Evento', index=False)
    freq_bigramas_ano_evento.to_excel(writer, sheet_name='Bigramas_Ano_Evento', index=False)
    formatar_workbook(writer)

print(f'Gerado: {arquivo_modelo}')
print(f'Gerado: {arquivo_resumos}')

## Relacionamentos sugeridos no Power BI
- `Artigos[id_artigo]` 1:N `Artigo_Eixo[id_artigo]`;
- `Eixos[id_eixo]` 1:N `Artigo_Eixo[id_eixo]`;
- `Eventos[evento]` 1:N `Artigos[evento]`;
- `Anos[ano]` 1:N `Artigos[ano]`.

Use direção de filtro simples das dimensões para as tabelas de fatos.